## Explore King County house sales data

A second real-estate dataset alongside the Ames one, run through the same discipline: look before
you model.

- Load & shape check — size, dtypes, duplicate rows.
- Target distribution — `price` and `log1p(price)`, skew.
- Missingness — is anything actually missing, and what do we do about it.
- Sanity checks — implausible values (e.g. a house with more bedrooms than a hotel).
- Dtype pass — catch numeric-coded categoricals (`zipcode`).
- Univariate distributions — histograms for key numerics.
- Numeric vs target — correlation table/heatmap, top movers.
- Geography vs target — this dataset has lat/long; Ames didn't.
- Outliers — data-entry errors that would mislead a model.
- Prepare data — cleaning + feature engineering decisions, written as a function.
- Train the model — same three algorithms as the main app (`Linear Regression`, `Decision Tree`,
  `Random Forest`), same log-target convention, so results are comparable.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("data/kc_house_data.csv")

pd.set_option("display.max_columns", 100)
print("shape:", df.shape)


In [ ]:
df.head()


### 1. Shape, dtypes, duplicates

Unlike Ames, this dataset has no pre-split train/test — it's one table of historical sales.

In [ ]:
print("duplicate rows (excl id):", df.drop(columns="id").duplicated().sum())
print("duplicate ids:", df["id"].duplicated().sum())
df.dtypes


**Findings:** no fully-duplicated rows, but 177 repeated `id`s. That's not a data bug — King County
house `id`s repeat when the *same house* sold more than once in the time window covered by the data
(each row is a sale, not a house). Keep both sales; they're independent observations of the market at
different points in time.


## 2. Target (`price`) distribution


In [ ]:
log_price = np.log1p(df["price"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["price"], bins=50)
axes[0].set_title("price")
axes[0].set_xlabel("price ($)")
axes[0].set_ylabel("Count")
axes[1].hist(log_price, bins=50)
axes[1].set_title("log1p(price)")
axes[1].set_xlabel("log1p(price)")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

print("price          skew: %.3f" % df["price"].skew())
print("log1p(price)   skew: %.3f" % log_price.skew())


**Findings:** `price` is heavily right-skewed (skew ≈ 4.0) — a long tail of expensive waterfront/
downtown properties. `log1p(price)` is much closer to symmetric. Same call as the Ames project: train
on `log1p(price)`, report RMSE on the log scale.


## 3. Missingness


In [ ]:
missing = df.isna().sum()
missing[missing > 0]


**Findings:** only `price` has missing values (4 rows), and nothing else. This is the opposite
situation from Ames, which had heavy, structured missingness across many columns (`NA` meaning "no
garage", etc.) — here there's no missingness pattern to model, just a handful of rows with no label at
all. Those rows are useless for supervised training and get dropped in `prepare data` below.


## 4. Descriptive stats & sanity checks


In [ ]:
df["bedrooms"].value_counts().sort_index()


**Findings:** bedroom counts climb smoothly from 0 to 11, then jump straight to a single house with
**33 bedrooms** on only 1,620 sqft of living space — physically implausible, almost certainly a typo
for 3. This is the King County dataset's equivalent of Ames's known `OUTLIER_IDS`: one row a model
should never be trained on. Confirmed below.


In [ ]:
df[df["bedrooms"] >= 10][["id", "bedrooms", "bathrooms", "sqft_living", "price"]]


## 5. Dtype pass


In [ ]:
print("zipcode distinct values:", df["zipcode"].nunique())
df[["waterfront", "view", "condition", "grade"]].describe()


**Findings:**
- `waterfront`, `view`, `condition`, `grade` are small integer *ordinal* scales (more is "better") —
  fine to leave as numeric, same reasoning as `OverallQual`/`OverallCond` in Ames.
- `zipcode` is a 70-level *nominal* code stored as `int64` — same trap as Ames's `MSSubClass`. Left
  numeric, a model would treat zipcode 98004 as "bigger than" 98002, which is meaningless. Cast to
  string before the `ColumnTransformer` split so it goes through `OneHotEncoder`.
- `date` (e.g. `"20141013T000000"`) is a sale timestamp, not a feature by itself — useful only once
  turned into something like sale year (see feature engineering below).


## 6. Univariate distributions


In [ ]:
key_numeric = ["sqft_living", "sqft_lot", "grade", "bathrooms", "yr_built", "lat"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), key_numeric):
    ax.hist(df[col], bins=40)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


**Findings:** `sqft_living` and `sqft_lot` are right-skewed like their Ames counterparts
(`GrLivArea`, `LotArea`) — a few very large properties. `grade` and `bathrooms` look roughly normal.
`yr_built` spans a full century of housing stock. Nothing here changes the modeling plan.


## 7. Numeric features vs target


In [ ]:
numeric_cols = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "waterfront", "view",
    "condition", "grade", "sqft_above", "sqft_basement", "yr_built", "yr_renovated",
    "lat", "long", "sqft_living15", "sqft_lot15",
]
target_corr = df[numeric_cols + ["price"]].corr()["price"].drop("price").sort_values(ascending=False)
target_corr


In [ ]:
top_features = target_corr.abs().sort_values(ascending=False).head(8).index.tolist()
corr_matrix = df[top_features + ["price"]].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation heatmap: top price movers")
plt.tight_layout()
plt.show()


**Findings:** `sqft_living` (0.70) and `grade` (0.67) are the strongest single predictors, echoing
Ames's `GrLivArea`/`OverallQual` pattern almost exactly — square footage and a quality/grade score
dominate price in both cities. `sqft_above` and `sqft_living15` are highly correlated with
`sqft_living` itself (near-redundant, same "not a problem for tree models or L2" note as Ames).
`condition` and `long` barely move price at all on their own.


## 8. Geography vs target

Ames didn't ship coordinates; this dataset does. Worth one look before treating `lat`/`long` as plain
numeric features.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(df["long"], df["lat"], c=np.log1p(df["price"]), cmap="viridis", s=4, alpha=0.5)
ax.set_xlabel("long")
ax.set_ylabel("lat")
ax.set_title("Sale locations, colored by log1p(price)")
fig.colorbar(sc, ax=ax, label="log1p(price)")
plt.tight_layout()
plt.show()


**Findings:** price is clearly clustered geographically — a bright band around downtown
Seattle/waterfront, cheaper further out. `lat`/`long` carry real signal, not noise, and are worth
keeping as plain numeric features (trees can carve up 2D space; linear models get a weaker, coarser
version of the same signal).


## 9. Prepare data


In [ ]:
def prepare_data(raw: pd.DataFrame) -> pd.DataFrame:
    """Clean + feature-engineer King County sales, mirroring app/services/preprocessing.py's shape."""
    out = raw.copy()

    # clean: rows with no label are useless; the 33-bedroom row is a data-entry error
    out = out.dropna(subset=["price"])
    out = out[out["bedrooms"] < 30]

    # engineer: age and renovation status are more informative than raw yr_built/yr_renovated
    sale_year = out["date"].str[:4].astype(int)
    out["house_age"] = sale_year - out["yr_built"]
    out["was_renovated"] = (out["yr_renovated"] > 0).astype(int)

    # dtype fix: zipcode is nominal, not ordinal
    out["zipcode"] = out["zipcode"].astype(str)

    return out.drop(columns=["date", "yr_renovated"])


prepared = prepare_data(df)
prepared[["house_age", "was_renovated", "zipcode"]].head()


**Feature engineering rationale** (same "combine what the model would otherwise have to
reconstruct" logic as `TotalSF`/`HouseAge`/`TotalBath` in the main app):

- **`house_age`** = sale year − `yr_built`. The model doesn't need two raw years and the arithmetic
  between them; it needs "how old was this house when it sold."
- **`was_renovated`** = whether `yr_renovated` is nonzero. 95.7% of rows are `0` ("never renovated"),
  not a missing value — using the raw year would make "never renovated" look like the oldest possible
  renovation, which is backwards. A yes/no flag is the honest encoding of what this column means.


## 10. Train the model


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

MODELS = {
    "Linear Regression": (LinearRegression(), {}),
    "Decision Tree": (
        DecisionTreeRegressor(random_state=42),
        {"model__max_depth": [4, 6, 8], "model__min_samples_leaf": [5, 10]},
    ),
    "Random Forest": (
        RandomForestRegressor(n_estimators=50, random_state=42),
        {"model__max_depth": [6, 10]},
    ),
}

y = np.log1p(prepared["price"])
X = prepared.drop(columns=["id", "price"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

results = {}
for name, (estimator, grid) in MODELS.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    if grid:
        search = GridSearchCV(pipe, grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=1)
        search.fit(X_train, y_train)
        best, params = search.best_estimator_, search.best_params_
    else:
        pipe.fit(X_train, y_train)
        best, params = pipe, None
    rmse = root_mean_squared_error(y_test, best.predict(X_test))
    results[name] = {"rmse_log": round(float(rmse), 4), "best_params": params}

pd.DataFrame(results).T


**Results** (RMSE on `log1p(price)`, lower is better):

| Model | RMSE (log) |
|---|---|
| Linear Regression | 0.1832 |
| Decision Tree | 0.2273 |
| Random Forest | 0.1850 |

**Findings:** Linear Regression and Random Forest land close together and both clearly beat a single
Decision Tree, which overfits/underfits depending on depth even after tuning. That mirrors the general
lesson from the Ames project: a single tree is rarely competitive, and there's no need to reach for
gradient boosting on a dataset this size (21.6k rows, similar to Ames's 1.5k in spirit) — a plain
linear model already does nearly as well as an ensemble once the target is log-transformed and the one
nominal high-cardinality column (`zipcode`) is one-hot encoded correctly.


## Summary

This was a from-scratch EDA + prepare + train pass on a second, unrelated real-estate dataset (King
County, WA house sales — 21.6k rows, 21 raw columns), kept separate from `app/services/` so it
doesn't touch the graded Ames pipeline or its deadline. The exercise: the *process* generalizes even
though the columns don't — validate → clean → engineer → train, same target-log-transform decision,
same "cast nominal codes to string before one-hot" fix, same "make the model's arithmetic explicit as
a feature" trick, on a dataset with completely different columns, scale, and quirks (repeat-sale ids,
almost no missingness, real lat/long instead of a neighborhood label).
